# Interacting with the OpenAI API


Load the secret as an environmental variable

In [ ]:
# Access your secret keys via
from google.colab import userdata
# The name of your secret must match `OPENAI_API_KEY`
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# Import OpenAI API and set up the key
from openai import OpenAI
client = OpenAI(api_key=(OPENAI_API_KEY))

## AI Calculator Bot with Comparison

This bot performs repeated multiplications. For each calculation, it uses GPT-4.1 to obtain a result and then compares this result with the correct calculation performed in Python. The bot will indicate whether the AI gave the correct answer or not.

In [ ]:
from google.colab import userdata
from openai import OpenAI

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=(OPENAI_API_KEY))

print("Starting AI calculator bot.")
print("Enter 'exit' at any time to stop.")

iteration_count = 0
errors = 0 # Initialize error counter
while True:
    user_input = input(f"\nIteration {iteration_count + 1}: Enter a mathematical expression to calculate: ")
    if user_input.lower() in ['quit', 'exit']:
        break

    try:
        # Replace '^' with '**' for proper python exponentiation
        python_expression = user_input.replace('^', '**')
        correct_result = eval(python_expression)
    except Exception as e:
        print(f"  Error evaluating Python expression: {e}")
        print("  Please enter a valid mathematical expression.")
        continue

    # Make API call to GPT-4.1 for calculation
    response = client.responses.create(
        model="gpt-4.1-nano",
        temperature=0,
        top_p=0.0,
        max_output_tokens=50,
        input=[
            {
                "role": "developer",
                "content": (
                    "You are a calculator. Evaluate the arithmetic expression the user gives you"
                    "and reply with the final number only: no words, no steps, no units."
                    "Use standard order of operations."
                    "If the input is not a math expression, reply exactly: ERROR"
                )
            },
            {
                "role": "user",
                "content": user_input
            }
        ]
    )

    ai_result_str = response.output_text.strip()

    try:
        ai_result = float(ai_result_str) # Use float for potentially non-integer results
        print(f"  Expression: {user_input}")
        print(f"  AI Result: {ai_result}")
        print(f"  Correct Result: {correct_result}")

        # Compare results, allowing for floating point inaccuracies
        if abs(ai_result - correct_result) < 1e-9: # Small tolerance for float comparison
            print("  Status: Ha ! AI got the correct answer!")
        else:
            errors += 1 # Increment error count on incorrect answer
            if errors == 1:
                print("  Status: Wow, how do humans deal with math ?")
            elif errors == 2:
                print("  Status: Oh no, another error... I guess I suck at math.")
            else:
                print("  Status: I give up. Apparently, calculating is not my strength.")
    except ValueError:
        errors += 1 # Also count AI returning non-numeric as an error
        print(f"  Expression: {user_input}")
        print(f"  AI did not return a valid number: '{ai_result_str}'")

    iteration_count += 1

print("Experiment finished.")

Starting AI calculator bot.
Enter 'exit' at any time to stop.

Iteration 1: Enter a mathematical expression to calculate: 5^5
  Expression: 5^5
  AI Result: 3125.0
  Correct Result: 3125
  Status: Ha ! AI got the correct answer!

Iteration 2: Enter a mathematical expression to calculate: 8*8
  Expression: 8*8
  AI Result: 64.0
  Correct Result: 64
  Status: Ha ! AI got the correct answer!

Iteration 3: Enter a mathematical expression to calculate: 8^8
  Expression: 8^8
  AI Result: 16777216.0
  Correct Result: 16777216
  Status: Ha ! AI got the correct answer!

Iteration 4: Enter a mathematical expression to calculate: 125^125
  Expression: 125^125
  AI did not return a valid number: 'ERROR'

Iteration 5: Enter a mathematical expression to calculate: 15^15
  Expression: 15^15
  AI Result: 4.378938903808594e+17
  Correct Result: 437893890380859375
  Status: Ha ! AI got the correct answer!

Iteration 6: Enter a mathematical expression to calculate: 25^25
  Expression: 25^25
  AI Result: 